# 05. E-commerce Landing Page A/B Test

This extension evaluates whether a redesigned landing page improves purchase conversion. It complements the transaction analysis: the A/B module addresses visitor-to-purchase conversion, while the UCI transaction module addresses post-purchase retention and customer value. The two public datasets are independent and are not joined at user level.

## 1. Experiment design

- **Control:** existing landing page (`old_page`)
- **Treatment:** redesigned landing page (`new_page`)
- **Primary metric:** visitor conversion rate
- **Null hypothesis:** treatment conversion equals control conversion
- **Alternative hypothesis:** treatment conversion differs from control conversion
- **Significance level:** 5%

A two-sided test is used because the redesign could improve or harm conversion.

In [ ]:
from pathlib import Path
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from scipy.stats import norm

pd.set_option('display.float_format',lambda x:f'{x:,.4f}')
plt.style.use('seaborn-v0_8-whitegrid')
repo_root=Path.cwd()
if repo_root.name=='notebooks': repo_root=repo_root.parent
data_path=repo_root/'data'/'raw'/'ab_data.csv'
output_dir=repo_root/'data'/'processed'; image_dir=repo_root/'images'
image_dir.mkdir(exist_ok=True)
raw=pd.read_csv(data_path,parse_dates=['timestamp'])
print(f'Raw rows: {len(raw):,}; unique users: {raw.user_id.nunique():,}')
display(raw.head())

## 2. Assignment and data-quality checks

A valid record must show the page assigned to that experiment group. Each user must contribute only one observation to preserve independence.

In [ ]:
quality=pd.Series({
 'Rows':len(raw),
 'Missing values':int(raw.isna().sum().sum()),
 'Exact duplicates':int(raw.duplicated().sum()),
 'Duplicate user rows':int(raw.user_id.duplicated().sum()),
 'Control shown new page':int(((raw.group=='control')&(raw.landing_page=='new_page')).sum()),
 'Treatment shown old page':int(((raw.group=='treatment')&(raw.landing_page=='old_page')).sum()),
})
display(quality.to_frame('count'))
valid_assignment=((raw.group=='control')&(raw.landing_page=='old_page'))|((raw.group=='treatment')&(raw.landing_page=='new_page'))
ab=raw.loc[valid_assignment].sort_values('timestamp').drop_duplicates('user_id',keep='first').copy()
ab['experiment_date']=ab.timestamp.dt.date
assert ab.user_id.is_unique
assert ((ab.group=='control')==(ab.landing_page=='old_page')).all()
print(f'Clean experiment rows: {len(ab):,}; removed: {len(raw)-len(ab):,}')

## 3. Validate traffic allocation and calculate conversion

In [ ]:
summary=(ab.groupby('group').agg(Visitors=('user_id','nunique'),Conversions=('converted','sum'),ConversionRate=('converted','mean')).reindex(['control','treatment']))
summary['TrafficShare']=summary.Visitors/summary.Visitors.sum()
display(summary)
control=summary.loc['control']; treatment=summary.loc['treatment']
absolute_effect=treatment.ConversionRate-control.ConversionRate
relative_effect=absolute_effect/control.ConversionRate
print(f'Absolute effect: {absolute_effect:.4%} ({absolute_effect*100:.3f} percentage points)')
print(f'Relative effect: {relative_effect:.2%}')

## 4. Two-proportion z-test and 95% confidence interval

In [ ]:
x_c,n_c=control.Conversions,control.Visitors
x_t,n_t=treatment.Conversions,treatment.Visitors
p_c,p_t=control.ConversionRate,treatment.ConversionRate
pooled=(x_c+x_t)/(n_c+n_t)
pooled_se=math.sqrt(pooled*(1-pooled)*(1/n_c+1/n_t))
z_stat=(p_t-p_c)/pooled_se
p_value=2*(1-norm.cdf(abs(z_stat)))
unpooled_se=math.sqrt(p_c*(1-p_c)/n_c+p_t*(1-p_t)/n_t)
ci_low=absolute_effect-norm.ppf(.975)*unpooled_se
ci_high=absolute_effect+norm.ppf(.975)*unpooled_se
result=pd.Series({'Control conversion':p_c,'Treatment conversion':p_t,'Absolute effect':absolute_effect,'Relative effect':relative_effect,'Z statistic':z_stat,'P value':p_value,'95% CI lower':ci_low,'95% CI upper':ci_high})
display(result.to_frame('value'))
decision='Reject H0' if p_value<.05 else 'Fail to reject H0'
print(f'Decision at alpha=0.05: {decision}')

## 5. Visualize effect size and stability

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
bars=ax.bar(['Control — old page','Treatment — new page'],[p_c,p_t],color=['#93C5FD','#2563EB'],width=.58)
ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_ylabel('Conversion rate'); ax.set_title('Landing Page Conversion Rate')
ax.set_ylim(0,max(p_c,p_t)*1.22)
for bar,value in zip(bars,[p_c,p_t]): ax.text(bar.get_x()+bar.get_width()/2,value+.002,f'{value:.2%}',ha='center',weight='bold')
plt.tight_layout(); plt.savefig(image_dir/'10_ab_conversion_rates.png',dpi=180,bbox_inches='tight'); plt.show()

daily=(ab.groupby(['experiment_date','group']).converted.mean().unstack())
ax=daily.plot(figsize=(11,5.5),color=['#93C5FD','#2563EB'],linewidth=2,marker='o',markersize=3)
ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.set_title('Daily Conversion Rate by Experiment Group'); ax.set_xlabel('Experiment date'); ax.set_ylabel('Conversion rate')
plt.tight_layout(); plt.savefig(image_dir/'11_ab_daily_conversion.png',dpi=180,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(8,3.8)); ax.errorbar(absolute_effect*100,0,xerr=[[ (absolute_effect-ci_low)*100 ],[ (ci_high-absolute_effect)*100 ]],fmt='o',color='#2563EB',ecolor='#2563EB',capsize=6,markersize=8)
ax.axvline(0,color='#EF4444',linestyle='--',linewidth=1.5); ax.set_yticks([]); ax.set_xlabel('Treatment effect (percentage points)'); ax.set_title('Estimated Conversion Effect with 95% Confidence Interval')
plt.tight_layout(); plt.savefig(image_dir/'12_ab_effect_confidence_interval.png',dpi=180,bbox_inches='tight'); plt.show()

## 6. Business recommendation

Statistical significance is not the only decision input. The direction and confidence interval matter: if the interval includes zero, the data do not establish that the redesign improves conversion. The team should avoid a full rollout, investigate why the new page did not improve behavior, and test a more targeted hypothesis rather than treating a neutral experiment as a successful launch.

In [ ]:
ab.to_csv(output_dir/'ab_test_clean.csv.gz',index=False,compression='gzip')
summary.reset_index().to_csv(output_dir/'ab_test_summary.csv',index=False)
pd.DataFrame([{'control_conversion':p_c,'treatment_conversion':p_t,'absolute_effect':absolute_effect,'relative_effect':relative_effect,'z_statistic':z_stat,'p_value':p_value,'ci_low':ci_low,'ci_high':ci_high,'decision':decision}]).to_csv(output_dir/'ab_test_result.csv',index=False)
print('A/B TEST INSIGHT SNAPSHOT')
print(f'1. Clean sample: {len(ab):,} unique users.')
print(f'2. Control: {p_c:.2%}; treatment: {p_t:.2%}.')
print(f'3. Effect: {absolute_effect*100:.3f} pp; 95% CI [{ci_low*100:.3f}, {ci_high*100:.3f}] pp.')
print(f'4. p-value: {p_value:.4f}; recommendation: do not roll out based on this experiment.')